Código criado por mim, mas seguindo rigidamente o exemplo do site. O cenário são algortimos de grafos usado em programção competitiva. Inicialmente tinha feito com a base de dados em português, mas o resultado saiu totalmente errado, provavelmente por que a configuração do TextEmbedding está para inglês. O resultado atual também é ruim, pois dijkistra não serve para aresta negativa.

Assim como no código anterior, a primeira sessão cria uma instância do QdrantClient para estabalecer a conexão com o qdrant cloud, usando a url e api_key.

In [ ]:
from qdrant_client import QdrantClient

# connect to Qdrant Cloud
client = QdrantClient(
    url="url aqui", 
    api_key="chave aqui",
)

Criação da coleção de itens, onde foi dado o nome de algorithms, pois serão armazenados apenas algoritmos de grafos. A configuração segue a mesma, vetor de 384 dimensões e usando a distância do cosseno.

In [14]:
from qdrant_client.models import Distance, VectorParams

# create collection
client.create_collection(
    collection_name="algorithms",
    vectors_config=VectorParams(size=384, distance=Distance.COSINE),
    timeout=30,
)

True

Parte mais crucial do programa. É instânciado o model, que tecebe as informações de como serializar as informações para o tipo definido no vectors_config. No algo_data estão as informações acerca dos algoritmos de grafo. A estrutura de serialização é a mesma: no embeddings são salvos o nome e a descrição do algoritmo já transformado para o vetor de 384 dimensões. É feito um for para percorrer todos os valores do algo_data e do embeddings, organizando tudo no point que posteriormente é adicionado em points. O client.upsert serve para mandar as informações adicionadas no points para o qdrant cloud para que ele possa ter a base de dados para comparar a query depois.

In [ ]:
from qdrant_client.models import PointStruct
from fastembed import TextEmbedding

# load the embedding model
model = TextEmbedding('BAAI/bge-small-en-v1.5')

# Base de conhecimento de grafos
algo_data = [
    ("DFS (Depth First Search)", "Graph traversal used for detecting cycles, topological sorting, and finding connected components.", "O(V + E)", "Travessia"),
    ("BFS (Breadth First Search)", "Traversal algorithm that finds the shortest path in unweighted graphs.", "O(V + E)", "Travessia"),
    ("Dijkstra", "Finds the shortest path from a source to all vertices in graphs with non-negative edge weights.", "O(E log V)", "Caminho Mínimo"),
    ("Bellman-Ford", "Computes shortest paths from a single source even with negative edge weights and detects negative cycles.", "O(V * E)", "Caminho Mínimo"),
    ("Floyd-Warshall", "Dynamic programming algorithm for finding shortest paths between all pairs of vertices.", "O(V^3)", "Caminho Mínimo entre todos os pares"),
    ("Segment Tree", "A tree data structure used for storing information about intervals or segments, allowing efficient range queries.", "O(log N)", "Estrutura de Dados")
]

# embedding generator
points = []
embeddings = model.embed([f"{a[0]} {a[1]}" for a in algo_data])

for i, embedding in enumerate(embeddings):
    points.append(PointStruct(
        id=i,
        vector=embedding.tolist(),
        payload={
            "name": algo_data[i][0],
            "description": algo_data[i][1],
            "complexity": algo_data[i][2],
            "category": algo_data[i][3]
        }
    ))


client.upsert(collection_name="algorithms", points=points)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

Execução da query_text de fato. Nela, pede um algoritmo para encontrar cilo ou caminhos mínimso em grados negativos, sendo que o mais usado pra isso é o Bellman-Ford. Query_text armazena a pergunta, query_vector recebe a pergunta serializada para vetor ed 384 dimensões, o client.query_points executa a busca dos 3 mais similares e os armazena em results. No for abaixo apenas percorre os resultados e os mostra.

In [ ]:
query_text = "how to find cycles or shortest path in negative graphs"

# Gerar vetor da pergunta
query_vector = next(iter(model.embed(query_text)))

# Buscar os 3 resultados mais próximos
results = client.query_points(
    collection_name="algorithms",
    query=query_vector,
    limit=3
)

# Exibição dos resultados encontrados
for res in results.points:
    print(f"Algoritmo Sugerido: {res.payload['name']}")
    print(f"Complexidade: {res.payload['complexity']}")
    print(f"Score: {res.score}")
    print("---")

Algoritmo Sugerido: Dijkstra
Complexidade: O(E log V)
Score: 0.81600684
---
Algoritmo Sugerido: BFS (Breadth First Search)
Complexidade: O(V + E)
Score: 0.81236935
---
Algoritmo Sugerido: Bellman-Ford
Complexidade: O(V * E)
Score: 0.80140585
---
